# IEEE 57-Bus DC-OPF with Line Switching and Angle Bounds
**Author:** Jewook Park  
**Date:** October 2025  
**Description:** DC-OPF with line switching and voltage angle bounds (±30°) for IEEE 57-bus system


## Setup and Data Loading


In [1]:
import os
os.environ["GRB_LICENSE_FILE"] = "/Users/a/Desktop/VIP/sc-opf/API key/gurobi.lic"
import re
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
print("Libraries loaded successfully!")


Libraries loaded successfully!


In [2]:
def extract_matrix_block(lines, varname):
    in_block = False
    matrix_lines = []
    for line in lines:
        if line.strip().startswith(f"{varname} = ["):
            in_block = True
            continue
        if in_block:
            if line.strip().startswith("];"):
                break
            clean = re.sub(r'%.*', '', line).strip().rstrip(';')
            if clean:
                matrix_lines.append(clean)
    matrix_data = []
    for line in matrix_lines:
        row = [float(x) for x in line.split()]
        matrix_data.append(row)
    return np.array(matrix_data)

path = 'data/pglib_opf_case57_ieee.m'
with open(path, 'r') as f:
    lines = f.readlines()

bus = extract_matrix_block(lines, 'mpc.bus')
branch = extract_matrix_block(lines, 'mpc.branch')
gen = extract_matrix_block(lines, 'mpc.gen')
gencost = extract_matrix_block(lines, 'mpc.gencost')

bus_df = pd.DataFrame(bus, columns=['bus_i','type','Pd','Qd','Gs','Bs','area','Vm','Va','baseKV','zone','Vmax','Vmin'])
gen_df = pd.DataFrame(gen, columns=['bus','Pg','Qg','Qmax','Qmin','Vg','mBase','status','Pmax','Pmin'])
branch_df = pd.DataFrame(branch, columns=['fbus','tbus','r','x','b','rateA','rateB','rateC','ratio','angle','status','angmin','angmax'])
gencost_df = pd.DataFrame(gencost, columns=['model','startup','shutdown','n','c2','c1','c0'])

print(f"Loaded: {len(bus_df)} buses, {len(branch_df)} branches, {len(gen_df)} generators")


Loaded: 57 buses, 80 branches, 7 generators


## DC-OPF Model with Line Switching and Angle Bounds


In [3]:
def solve_dc_opf_anglebound(bus_df, gen_df, branch_df, gencost_df, verbose=True):
    """DC-OPF with line switching and angle bounds"""
    model = gp.Model("DC-OPF-AngleBound-Case57")
    if not verbose:
        model.Params.OutputFlag = 0
    
    ref_bus = bus_df[bus_df['type'] == 3]['bus_i'].values[0]
    
    # Angle bounds: ±30 degrees (0.524 radians)
    MAX_ANGLE_RAD = 0.524
    
    # Calculate Big-M
    max_susceptance = (1.0 / branch_df['x']).max()
    max_capacity = branch_df['rateA'].max()
    M = max(2 * MAX_ANGLE_RAD * max_susceptance, max_capacity * 2)
    
    print(f"Angle bounds: ±{MAX_ANGLE_RAD * 180 / np.pi:.1f}°")
    print(f"Big-M: {M:.2f}")
    
    # Variables with angle bounds
    Pg = model.addVars(gen_df.index, lb=0, name="Pg")
    theta = model.addVars(bus_df['bus_i'], lb=-MAX_ANGLE_RAD, ub=MAX_ANGLE_RAD, name="theta")
    P_branch = model.addVars(branch_df.index, lb=-GRB.INFINITY, name="P_branch")
    z = model.addVars(branch_df.index, vtype=GRB.BINARY, name="z")
    
    # Objective
    obj = gp.QuadExpr()
    for idx, row in gencost_df.iterrows():
        obj += row['c2'] * Pg[idx]*Pg[idx] + row['c1'] * Pg[idx] + row['c0']
    model.setObjective(obj, GRB.MINIMIZE)
    
    # Constraints
    model.addConstr(theta[ref_bus] == 0, name="ref_bus")
    
    for idx, row in gen_df.iterrows():
        model.addConstr(Pg[idx] >= row['Pmin'], name=f"Pg_min_{idx}")
        model.addConstr(Pg[idx] <= row['Pmax'], name=f"Pg_max_{idx}")
    
    for idx, row in branch_df.iterrows():
        fbus, tbus, x = row['fbus'], row['tbus'], row['x']
        B = 1.0 / x
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) <= M * (1 - z[idx]), name=f"flow_up_{idx}")
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) >= -M * (1 - z[idx]), name=f"flow_lo_{idx}")
        model.addConstr(P_branch[idx] <= M * z[idx], name=f"off_up_{idx}")
        model.addConstr(P_branch[idx] >= -M * z[idx], name=f"off_lo_{idx}")
    
    for idx, row in branch_df.iterrows():
        rateA = row['rateA']
        if rateA > 0:
            model.addConstr(P_branch[idx] <= rateA * z[idx], name=f"cap_up_{idx}")
            model.addConstr(P_branch[idx] >= -rateA * z[idx], name=f"cap_lo_{idx}")
    
    for idx, row in bus_df.iterrows():
        bus, Pd = row['bus_i'], row['Pd']
        gen_at_bus = gen_df[gen_df['bus'] == bus].index.tolist()
        gen_P = gp.quicksum(Pg[g] for g in gen_at_bus) if gen_at_bus else 0
        branch_out = gp.quicksum(P_branch[br] for br in branch_df[branch_df['fbus'] == bus].index)
        branch_in = gp.quicksum(P_branch[br] for br in branch_df[branch_df['tbus'] == bus].index)
        model.addConstr(gen_P - Pd == branch_out - branch_in, name=f"balance_{bus}")
    
    model.optimize()
    return model, Pg, theta, P_branch, z


In [4]:
model, Pg, theta, P_branch, z = solve_dc_opf_anglebound(bus_df, gen_df, branch_df, gencost_df)


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2725033
Academic license 2725033 - for non-commercial use only - registered to jp___@gatech.edu
Angle bounds: ±30.0°
Big-M: 3234.00
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Academic license 2725033 - for non-commercial use only - registered to jp___@gatech.edu
Optimize a model with 552 rows, 224 columns and 1462 nonzeros
Model fingerprint: 0x416af07c
Variable types: 144 continuous, 80 integer (80 binary)
Coefficient statistics:
  Matrix range     [7e-01, 3e+03]
  Objective range  [2e+01, 4e+01]
  Bounds range     [5e-01, 1e+00]
  RHS range        [2e+00, 3e+03]
Presolve removed 211 rows and 66 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 10 available processors)

Solution count 0



In [5]:
if model.status == GRB.OPTIMAL:
    print("=" * 70)
    print("IEEE 57-BUS DC-OPF WITH LINE SWITCHING & ANGLE BOUNDS")
    print("Author: Jewook Park")
    print("=" * 70)
    print(f"Cost: ${model.objVal:.2f}")
    print(f"Total Gen: {sum(Pg[i].X for i in gen_df.index):.2f} MW")
    print(f"Total Load: {bus_df['Pd'].sum():.2f} MW")
    
    lines_on = sum(z[i].X > 0.5 for i in branch_df.index)
    print(f"\nLines ON: {lines_on}/{len(branch_df)}")
    
    # Angle difference analysis
    angle_diffs = []
    for idx, row in branch_df.iterrows():
        if z[idx].X > 0.5:
            angle_diff = abs(theta[row['fbus']].X - theta[row['tbus']].X) * 180 / np.pi
            angle_diffs.append(angle_diff)
    print(f"Max Angle Difference: {max(angle_diffs):.2f}°")
    print(f"Avg Angle Difference: {np.mean(angle_diffs):.2f}°")
else:
    print(f"Failed: {model.status}")


Failed: 3
